# 第二章：把模型里的基础计算亲手写出来

[上一章](../ch01/ch01.ipynb) · [启动说明](../README.md) · 代码版本：`tutorial-02-v2`

上一章，我们写了生成循环，但下面这一步仍由 HF 完成：

```python
logits = model(input_ids=input_ids, use_cache=False).logits
```

现在往里面走一层：亲手写 **Embedding、Linear、RMSNorm、SiLU、MLP 和残差连接**。
每个组件先用能手算的小例子理解，再用真实模型的同一份输入和权重核对。

**本章结束，完整生成仍然调用 HF。** 我们先把零件写对，第五章才接成完整的手写模型。
第一章的 `generate.py` 在这两个版本之间没有变化。

| 原来交给框架 | 本章亲手写的计算 |
|---|---|
| Embedding | 用 token ID 取权重表中的一行 |
| Linear | 输入与权重矩阵相乘 |
| RMSNorm | 计算均方根，缩放向量 |
| SiLU / MLP | 非线性和三个 Linear 的组合 |
| MLP 残差分支 | `x + mlp(norm(x))` |

按 **Shift+Enter** 从头运行。前半章只用小数组；第7节才加载一次0.8B参考模型，采集结果后释放模型。
更改定义或输入后，请重启 kernel 并顺序重跑。不要和其他模型测试同时占用大量内存。

## 1. 先看清输入和输出

第一章模型两端的形状如下。H 是隐藏向量长度，V 是词表大小：

```text
IDs          Embedding           中间各层          最后 norm + Linear
[B,T]   →    [B,T,H]       →      [B,T,H]      →       [B,T,V]
整数         token 向量           上下文向量             logits
```

两个 `[B,T,H]` 形状相同，内容却不同：中间各层已经处理了上下文。
本章0.8B模型的 H=1024、V=248320；先用 H=3、V=5 看懂运算。

`mlx.core` 是 Apple Silicon 的数组运算库；我们的组件只依赖它。
PyTorch / NumPy 在本子里用于对照。`@dataclass` 自动生成保存权重的初始化方法，
`__call__` 则允许写成 `layer(x)`。

In [1]:
from dataclasses import dataclass
from importlib.metadata import version
from pathlib import Path
import gc
import sys

import mlx.core as mx
import numpy as np
import torch
import torch.nn.functional as F

for package in ['mlx', 'torch', 'transformers']:
    print(package + ':', version(package))

mlx: 0.32.3
torch: 2.14.1
transformers: 5.19.0


## 2. Embedding：按 ID 查一行

每行对应一个 token。先猜：取 ID 2、0，应该得到哪两行？

In [2]:
toy_weight = mx.array([
    [1., 0., 1.],  # ID 0
    [0., 1., 1.],  # ID 1
    [1., 1., 0.],  # ID 2
    [-1., 0., 1.], # ID 3
    [0., -1., 1.], # ID 4
])
toy_ids = mx.array([2, 0])
print('权重表 shape:', toy_weight.shape)

权重表 shape: (5, 3)


框架的等价调用是 `F.embedding(ids, weight)`。
我们把实际计算写出来，只需要 `weight[token_ids]`：

In [3]:
@dataclass
class Embedding:
    weight: mx.array                 # [vocab_size, hidden_size]

    def __call__(self, token_ids: mx.array) -> mx.array:
        return self.weight[token_ids]

In [4]:
toy_embedding = Embedding(toy_weight)
embedded = toy_embedding(toy_ids)
reference = F.embedding(
    torch.tensor([2, 0]), torch.tensor(np.array(toy_weight)),
)
print(np.array(embedded))
print('输出 shape:', embedded.shape)
np.testing.assert_array_equal(np.array(embedded), reference.numpy())

[[1. 1. 0.]
 [1. 0. 1.]]
输出 shape: (2, 3)


这里 `[2] → [2,3]`：2个 ID，每个取回3个数。若加 batch 维，就是 `[B,T] → [B,T,H]`。
ID 只负责选行；ID 接近不表示两个词意思接近。权重来自训练，推理时只读取它。

## 3. Linear：和每一行做内积

假设经过中间层后，最后一个位置的向量是 `h=[1,2,0]`。这里是人为设定的小例子，
**不能把初始 Embedding 直接当成完整模型处理后的向量。**

第 j 个候选的分数就是 h 与权重第 j 行的内积。例如 ID 2：

```text
h = [1,2,0]，W[2] = [1,1,0]
分数 = 1×1 + 2×1 + 0×0 = 3
```

下面先把5个候选分别算一遍。

In [5]:
hidden = mx.array([[1., 2., 0.]])
scores_one_by_one = [float(mx.sum(hidden[0] * row).item()) for row in toy_weight]
print('逐行内积:', scores_one_by_one)

逐行内积: [1.0, 2.0, 3.0, -1.0, -2.0]


框架用 `F.linear(x, weight)` 一次完成所有内积。我们写成矩阵乘：

```text
x [T,H]  @  W.T [H,V]  →  分数 [T,V]
```

权重存成 `[输出维度, 输入维度]`，所以要转置。这里的模型投影没有 bias。

In [6]:
@dataclass
class Linear:
    weight: mx.array                 # [out_features, in_features]

    def __call__(self, x: mx.array) -> mx.array:
        return x @ self.weight.T     # [..., in] → [..., out]

In [7]:
toy_head = Linear(toy_weight)
scores = toy_head(hidden)
reference = F.linear(
    torch.tensor(np.array(hidden)), torch.tensor(np.array(toy_weight)),
)
np.testing.assert_array_equal(np.array(scores), reference.numpy())
np.testing.assert_array_equal(np.array(scores[0]), scores_one_by_one)
print('所有候选分数:', np.array(scores))
print('greedy 选择 ID:', int(mx.argmax(scores[0]).item()))

所有候选分数: [[ 1.  2.  3. -1. -2.]]
greedy 选择 ID: 2


这个例子用同一张表做两件事：**输入端查行，输出端做内积**。
真实的0.8B模型也共享这两端的权重，称为 tied embeddings。
共享权重不会让两种运算变成同一种；其他模型也可以用独立的两张表。

## 4. RMSNorm：先控制向量的大小

只看一个向量 `[3,4]`，暂时令 epsilon 为0、最终缩放系数为1：

```text
[3,4] → 平方 [9,16] → 平均 12.5 → 开根号 3.5355...
  └──────────────────── 除以上面的数 → [0.8485,1.1314]
```

它不减均值。下面每一步都打印，观察归一化后平方的平均值。

In [8]:
norm_input = mx.array([[3., 4.]])
mean_square = mx.mean(norm_input * norm_input, axis=-1, keepdims=True)
root_mean_square = mx.sqrt(mean_square)
normalized = norm_input / root_mean_square
print('平方平均:', np.array(mean_square))
print('均方根:', np.array(root_mean_square))
print('归一化:', np.array(normalized))
print('归一化后的平方平均:', np.array(mx.mean(normalized * normalized)))

平方平均: [[12.5]]
均方根: [[3.535534]]
归一化: [[0.84852815 1.1313709 ]]
归一化后的平方平均: 1.0


实际计算还要加很小的 epsilon，避免零向量除以零，再乘每一维的可学习系数。
**这个 Qwen 模型保存的是 w，真正的缩放系数是 `1+w`。** 因此权重全0时，系数仍为1。

HF 参考计算展开后是：

```python
xf = x.float()
y = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + eps)
y = y * (1.0 + weight.float())
```

我们的计算用 MLX 写出相同步骤。`rsqrt` 就是开平方后的倒数。
先用 FP32 算均方值，最后转回输入 dtype；本章权重与输入统一为 FP32。

In [9]:
@dataclass
class RMSNorm:
    weight: mx.array                 # [hidden_size], stored as w for Qwen
    eps: float = 1e-6
    zero_centered: bool = True

    def __call__(self, x: mx.array) -> mx.array:
        # The explicit math is kept here before introducing fused kernels.
        xf = x.astype(mx.float32)
        mean_square = mx.mean(xf * xf, axis=-1, keepdims=True)
        normalized = xf * mx.rsqrt(mean_square + self.eps)
        scale = 1.0 + self.weight if self.zero_centered else self.weight
        return (normalized * scale).astype(x.dtype)

In [10]:
toy_norm = RMSNorm(mx.zeros(2), eps=0.)
norm_result = toy_norm(norm_input)
np.testing.assert_allclose(
    np.array(norm_result), np.array(normalized), rtol=0, atol=1e-6,
)
safe_norm = RMSNorm(mx.zeros(2))
print('w=0，实际系数1+w:', np.array(norm_result))
print('带 epsilon 处理零向量:', np.array(safe_norm(mx.zeros((1, 2)))))

w=0，实际系数1+w: [[0.84852815 1.1313709 ]]
带 epsilon 处理零向量: [[0. 0.]]


## 5. SiLU：加入非线性

连续做多个无 bias 的 Linear，仍可合并成一个矩阵乘。
SiLU 对每个元素计算 `x * sigmoid(x)`，使组合不再只是线性映射。

sigmoid 在0和1之间，但 SiLU 又乘了 x，因此 SiLU **可以为负，也可以大于1**。
先猜 `-2、0、2` 各自会变成什么，再运行。

In [11]:
def silu(x: mx.array) -> mx.array:
    return x * mx.sigmoid(x)

In [12]:
activation_input = mx.array([-2., 0., 2.])
activation_output = silu(activation_input)
print('输入:', np.array(activation_input))
print('sigmoid:', np.array(mx.sigmoid(activation_input)))
print('SiLU:', np.array(activation_output))
np.testing.assert_allclose(
    np.array(activation_output), F.silu(torch.tensor([-2., 0., 2.])).numpy(),
    rtol=0, atol=1e-6,
)

输入: [-2.  0.  2.]
sigmoid: [0.11920292 0.5        0.8807971 ]
SiLU: [-0.23840584  0.          1.7615942 ]


## 6. 组合成 MLP，再加回原输入

这个模型的 MLP 有三个 Linear。gate 和 up 是两条并行分支，相乘后交给 down：

```text
                      gate_proj → SiLU ─┐
输入 x（H维） ────────┤                   × → down_proj → 输出（H维）
                      up_proj ──────────┘
                           两分支都是 I 维
```

`*` 是对应元素相乘，不是 `@` 矩阵乘。gate 这条分支的数值不是概率，也不保证在0到1之间。
真实模型的 H=1024、I=3584；先让三个投影都是2×2单位矩阵，这样能直接看清中间值。

In [13]:
@dataclass
class MLP:
    gate_proj: Linear                # H → I
    up_proj: Linear                  # H → I
    down_proj: Linear                # I → H

    def __call__(self, x: mx.array) -> mx.array:
        gate = silu(self.gate_proj(x))
        value = self.up_proj(x)
        return self.down_proj(gate * value)

In [14]:
identity = Linear(mx.eye(2))
toy_mlp = MLP(identity, identity, identity)
branch_input = mx.array([[1., 2.]])
gate = silu(toy_mlp.gate_proj(branch_input))
value = toy_mlp.up_proj(branch_input)
print('gate 经过 SiLU:', np.array(gate))
print('up 分支:', np.array(value))
print('逐元素相乘:', np.array(gate * value))
print('MLP 输出:', np.array(toy_mlp(branch_input)))
expected = np.array(branch_input) ** 2 / (1 + np.exp(-np.array(branch_input)))
np.testing.assert_allclose(np.array(toy_mlp(branch_input)), expected, rtol=0, atol=1e-6)

gate 经过 SiLU: [[0.7310586 1.7615942]]
up 分支: [[1. 2.]]
逐元素相乘: [[0.7310586 3.5231884]]
MLP 输出: [[0.7310586 3.5231884]]


MLP 对每个位置的向量分别运算，不直接混合不同 token。跨位置的信息由 Attention / GDN 处理，
我们到后面两章再写。

一层里面还会把这条分支的结果加回原输入：`x + mlp(norm(x))`。
先观察每一步，不把它藏在一行里：

In [15]:
branch_norm = RMSNorm(mx.zeros(2))
normalized_branch = branch_norm(branch_input)
change = toy_mlp(normalized_branch)
residual_result = branch_input + change
print('原输入:', np.array(branch_input))
print('norm 后:', np.array(normalized_branch))
print('MLP 改变量:', np.array(change))
print('加回原输入:', np.array(residual_result))

原输入: [[1. 2.]]
norm 后: [[0.6324554 1.2649108]]
MLP 改变量: [[0.2612183 1.247792 ]]
加回原输入: [[1.2612183 3.247792 ]]


现在把三步封装成 `FeedForwardBlock`。这是 decoder 层的第二个残差分支，
不是整个 decoder 层：第一段还包含 Attention 或 GDN。

In [16]:
@dataclass
class FeedForwardBlock:
    """The second sub-block of a decoder layer: x + MLP(norm(x))."""
    norm: RMSNorm
    mlp: MLP

    def __call__(self, x: mx.array) -> mx.array:
        return x + self.mlp(self.norm(x))

In [17]:
toy_block = FeedForwardBlock(branch_norm, toy_mlp)
np.testing.assert_array_equal(
    np.array(toy_block(branch_input)), np.array(residual_result),
)

# 如果 down 投影全0，MLP 改变量为0，输出必须等于原始 x。
zero_down = Linear(mx.zeros((2, 2)))
zero_block = FeedForwardBlock(branch_norm, MLP(identity, identity, zero_down))
np.testing.assert_array_equal(
    np.array(zero_block(branch_input)), np.array(branch_input),
)
print('零分支保留原输入:', np.array(zero_block(branch_input)))

零分支保留原输入: [[1. 2.]]


## 7. 换上真实权重：比较的是同一个问题吗？

小例子过关了，接着用第一章那19个输入 ID 验证真实模型。
HF 完整跑一次，记录第一层组件的输入、输出，以及最后的隐藏向量和 logits。
**我们的组件拿到完全相同的输入和权重，再重算一次。**

```text
同一份输入与权重 ──→ HF 组件 ──→ 参考输出
               └──→ 本子里刚定义的组件 ──→ 我们的输出
                                               ↓
                                        比较形状和数值误差
```

采集辅助代码在 [hf_fixture.py](hf_fixture.py)，由冻结的 [check_layers.py](reference/check_layers.py)
整理而来。它只负责加载 HF、记录中间数组、移除 hook 和释放模型；不会导入手写组件。
下面所有重算都调用**你当前在本子中定义的类**。修改这些类后，检查结果也会变化。

这一步需要本机缓存中的固定0.8B权重；下载方式与第一章相同，见[启动说明](../README.md)。

In [18]:
here = Path.cwd().resolve()
chapter_dir = next(p for p in [here, here / 'ch02', here / 'notebooks/ch02']
                   if (p / 'hf_fixture.py').is_file())
if str(chapter_dir) not in sys.path:
    sys.path.insert(0, str(chapter_dir))
from hf_fixture import load_hf_fixture

DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'
if 'fixture' in globals():
    del fixture
gc.collect()
fixture = load_hf_fixture(device=DEVICE)
print('参考设备:', fixture['device'])
print('输入 ID 数:', len(fixture['prompt_ids']))
print('两端共享权重:', fixture['weights_shared'])
print('真实 Embedding 权重 shape:', fixture['embedding_weight'].shape)

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


参考设备: mps
输入 ID 数: 19
两端共享权重: True
真实 Embedding 权重 shape: (248320, 1024)


先把真实权重交给刚定义的类。输入端的表和输出头共享同一个 MLX 数组，
norm 的 epsilon 也从参考层读取。

In [19]:
embedding = Embedding(mx.array(fixture['embedding_weight']))
head = Linear(embedding.weight) if fixture['weights_shared'] else Linear(
    mx.array(fixture['head_weight'])
)
norms = {
    name: RMSNorm(mx.array(info['weight']), eps=info['eps'])
    for name, info in fixture['norms'].items()
}
mlp = MLP(*(Linear(mx.array(fixture['mlp_weights'][name]))
            for name in ['gate', 'up', 'down']))
feed_forward = FeedForwardBlock(norms['post_norm'], mlp)
captured = fixture['captured']
print('gate 权重:', mlp.gate_proj.weight.shape)
print('down 权重:', mlp.down_proj.weight.shape)

gate 权重: (3584, 1024)
down 权重: (1024, 3584)


比较函数先检查形状，再求最大的绝对误差。
查表应完全一致；矩阵乘和归约的浮点执行顺序可能不同，所以沿用原章节的数值阈值。
通过这些有限输入的检查，不等于证明所有输入都正确，也不等于完成了整模型替换。

In [20]:
errors = {}
def check(name, actual, expected, tolerance=1e-3):
    mx.eval(actual)  # MLX 延迟执行；在这里拿到实际计算结果
    actual_array = np.array(actual)
    assert actual_array.shape == expected.shape, name
    assert actual_array.dtype == expected.dtype == np.float32, name
    assert np.isfinite(actual_array).all() and np.isfinite(expected).all(), name
    error = float(np.max(np.abs(actual_array - expected)))
    assert error < tolerance, f'{name}: error={error}'
    errors[name] = error
    print(f'{name}: max_abs_error={error:.8g}')

**第一组：查表、三个 norm、第一层 MLP。** 每个都先单独核对，方便定位是哪一步有差异。

In [21]:
ids = mx.array([fixture['prompt_ids']])
check('embedding', embedding(ids), fixture['ref_embedding'], tolerance=1e-6)
for name, norm in norms.items():
    check(name, norm(mx.array(captured[name + '_in'])), captured[name + '_out'])
check('mlp', mlp(mx.array(captured['mlp_in'])), captured['mlp_out'])

embedding: max_abs_error=0
input_norm: max_abs_error=4.7683716e-07
post_norm: max_abs_error=1.9073486e-06
final_norm: max_abs_error=3.8146973e-06
mlp: max_abs_error=3.2782555e-07


**第二组：我们能否重算整个 MLP 残差分支？**

HF 那一段是：

```python
ref = x + hf_layer.mlp(hf_layer.post_attention_layernorm(x))
```

下面的 `x` 已经是 HF 完成前面 mixer 分支后的输入。这里只替换第二段，
不能直接把初始 Embedding 塞进来与完整层输出比较。

In [22]:
x = mx.array(captured['post_norm_in'])
normalized = norms['post_norm'](x)  # ① 我们的 RMSNorm
change = mlp(normalized)           # ② 我们的 MLP
ours = x + change                 # ③ 加回原始输入
check('feed_forward', ours, captured['layer_out'])
np.testing.assert_array_equal(np.array(feed_forward(x)), np.array(ours))
print('残差分支输出 shape:', ours.shape)

feed_forward: max_abs_error=4.1723251e-07
残差分支输出 shape: (1, 19, 1024)


**第三组：用我们自己的最后 norm 和输出头得到 logits。**
中间隐藏向量仍由 HF 提供；这是输出端的组合检查，不是从 IDs 开始的完整手写生成。

In [23]:
check('output_head', head(mx.array(captured['head_in'])), fixture['ref_logits'])
ours_logits = head(norms['final_norm'](mx.array(captured['final_norm_in'])))
check('final_norm_and_head', ours_logits, fixture['ref_logits'])
ours_ids = np.array(ours_logits).argmax(axis=-1)
ref_ids = fixture['ref_logits'].argmax(axis=-1)
np.testing.assert_array_equal(ours_ids, ref_ids)
next_id = int(ours_ids[0, -1])
assert next_id == fixture['expected_first_id']
print('logits shape:', ours_logits.shape)
print('全部位置的 argmax 一致:', True)
print('最后位置选出的 ID:', next_id, '（与第一章首 token 相同）')

output_head: max_abs_error=0
final_norm_and_head: max_abs_error=7.6293945e-06
logits shape: (1, 19, 248320)
全部位置的 argmax 一致: True
最后位置选出的 ID: 79852 （与第一章首 token 相同）


## 8. 本章结束，我们拥有什么？

| 能力 | 当前状态 |
|---|---|
| 六个基础组件及残差组合 | 已在本子里显式实现，用小例子和真实参考核对 |
| 从隐藏向量得到 logits | 手写最后 norm + 输出头，已独立核对 |
| 从 IDs 得到所有中间隐藏向量 | 仍由 HF 完成；Attention / GDN 等尚未实现 |
| 第一章的生成循环 | 保留，完整模型仍调用 HF |

所以这章不只写了一个 matmul。但这些组件暂时还没有接管完整模型。
所有类的初始定义来自 `tutorial-02-v2` 的 [layers.py](reference/layers.py)，
原检查结果也保留在 [历史记录](reference/examples/chapter02-v2-checks.json)。
新运行的误差以本子的实际输出为准，不要求与历史每一个小数完全相同。

## 9. 先预测，再改一处

1. Embedding 的 IDs 改成 `[0,2,0]`：输出形状和哪两行会相同？
2. RMSNorm 保存的 w 为0时，把 `1+w` 错写成 `w`，会发生什么？
3. 把残差里的原输入 x 错换为 `norm(x)`，零 down 投影的检查还能通过吗？

先在小例子动手，答案见[独立练习本](exercise-solutions.ipynb)。
下一章再学习 [Attention](../../docs/tutorial-v2/03-attention.md)，不在这一章继续展开。

运行下面的清理格，释放真实权重及中间数组。重新检查真实模型时，从第7节顺序执行；
结束学习也可以直接关闭 kernel。

In [24]:
for name in ['fixture', 'captured', 'embedding', 'head', 'norms', 'norm',
             'mlp', 'feed_forward', 'x', 'normalized', 'change', 'ours',
             'ours_logits', 'ours_ids', 'ref_ids']:
    globals().pop(name, None)
gc.collect()
mx.clear_cache()
print('已释放真实权重与中间数组。')

已释放真实权重与中间数组。

编排继续采用第一章的“短讲解、短代码、实际输出、独立练习”形式。
固定代码 commit：`f3b203ba44b135693bf0c6d6dbc82dd8072e9a63`；来源哈希见 [manifest](reference/manifest.json)。
本章参考模型快照与第一章一致，不做速度比较。